In [55]:
using SymPy

In [56]:
function sx(s)
    x, y = symbols("x y")
    diff(diff(s, y), y)
end
function sy(s)
    x, y = symbols("x y")
    diff(diff(s, x), x)
end
function sxy(s)
    x, y = symbols("x y")
    -diff(diff(s, x), y)
end
function ΔΔ(f)
    x, y = symbols("x y")
    diff(diff(diff(diff(f, x), x), x), x) + 2 * diff(diff(diff(diff(f, x), x), y), y) + diff(diff(diff(diff(f, y), y), y), y)
end
function Δ(f)
    x, y = symbols("x y")
    diff(diff(f, x), x) + diff(diff(f, y), y)
end

Δ (generic function with 1 method)

In [57]:
function airy_ansatz(deg)
    x, y = symbols("x y")
    coeffs = Sym[]
    ϕ = 0

    k = 0
    for i in 0:deg
        for j in 0:(deg-i)
            C = symbols("C$k")
            push!(coeffs, C)
            ϕ += C * x^i * y^j
            k += 1
        end
    end

    return ϕ, coeffs
end

airy_ansatz (generic function with 1 method)

In [58]:
function coeff_equations(expr)
    x, y = symbols("x y")
    p = sympy.Poly(expr, x, y)
    return p.coeffs()
end

coeff_equations (generic function with 1 method)

In [59]:
function coeff_equations(expr)
    x, y = symbols("x y")
    return collect(values(expr.as_poly(x, y).as_dict()))
end

coeff_equations (generic function with 1 method)

In [60]:
ϕ, coeffs = airy_ansatz(4)

(C0 + C1*y + C10*x^2*y + C11*x^2*y^2 + C12*x^3 + C13*x^3*y + C14*x^4 + C2*y^2 + C3*y^3 + C4*y^4 + C5*x + C6*x*y + C7*x*y^2 + C8*x*y^3 + C9*x^2, Sym[C0, C1, C2, C3, C4, C5, C6, C7, C8, C9, C10, C11, C12, C13, C14])

In [61]:
ΔΔϕ = ΔΔ(ϕ)

8⋅C₁₁ + 24⋅C₁₄ + 24⋅C₄

In [62]:
x, y = symbols("x y")
b, h = symbols("b h")
N, M, T, p, t, q = symbols("N M T p t q")

(N, M, T, p, t, q)

In [63]:
V = q * y

q⋅y

In [64]:
ΔV = Δ(V)

0

In [65]:
ν = symbols("ν")

ν

In [66]:
eqs = coeff_equations(ΔΔϕ + (1 - ν) * ΔV)

1-element Vector{Sym{PyCall.PyObject}}:
 8⋅C₁₁ + 24⋅C₁₄ + 24⋅C₄

In [67]:
σx = sx(ϕ) + V

       2                          2                          
2⋅C₁₁⋅x  + 2⋅C₂ + 6⋅C₃⋅y + 12⋅C₄⋅y  + 2⋅C₇⋅x + 6⋅C₈⋅x⋅y + q⋅y

In [68]:
σy = sy(ϕ) + V

                 2                                 2             
2⋅C₁₀⋅y + 2⋅C₁₁⋅y  + 6⋅C₁₂⋅x + 6⋅C₁₃⋅x⋅y + 12⋅C₁₄⋅x  + 2⋅C₉ + q⋅y

In [69]:
τxy = sxy(ϕ)

                              2                       2
-2⋅C₁₀⋅x - 4⋅C₁₁⋅x⋅y - 3⋅C₁₃⋅x  - C₆ - 2⋅C₇⋅y - 3⋅C₈⋅y 

In [70]:
expN = integrate(σy, (x, -b / 2, b / 2))
expN = expN(y => -h)
eqs = vcat(eqs, [expN])

2-element Vector{Sym{PyCall.PyObject}}:
                          8⋅C₁₁ + 24⋅C₁₄ + 24⋅C₄
 C14*b^3 + b*(-2*C10*h + 2*C11*h^2 + 2*C9 - h*q)

In [71]:
expM = integrate(x * σy, (x, -b / 2, b / 2))
expM = expM(y => -h)
eqs = vcat(eqs, [expM])

3-element Vector{Sym{PyCall.PyObject}}:
                          8⋅C₁₁ + 24⋅C₁₄ + 24⋅C₄
 C14*b^3 + b*(-2*C10*h + 2*C11*h^2 + 2*C9 - h*q)
                         b^3*(2*C12 - 2*C13*h)/4

In [72]:
expT = integrate(τxy, (x, -b / 2, b / 2))
expT = expT(y => -h)
eqs = vcat(eqs, [expT])

4-element Vector{Sym{PyCall.PyObject}}:
                          8⋅C₁₁ + 24⋅C₁₄ + 24⋅C₄
 C14*b^3 + b*(-2*C10*h + 2*C11*h^2 + 2*C9 - h*q)
                         b^3*(2*C12 - 2*C13*h)/4
        -C13*b^3/4 + b*(-C6 + 2*C7*h - 3*C8*h^2)

In [73]:
s1 = σx(x => b / 2)
eqs = vcat(eqs, coeff_equations(s1))

7-element Vector{Sym{PyCall.PyObject}}:
                          8⋅C₁₁ + 24⋅C₁₄ + 24⋅C₄
 C14*b^3 + b*(-2*C10*h + 2*C11*h^2 + 2*C9 - h*q)
                         b^3*(2*C12 - 2*C13*h)/4
        -C13*b^3/4 + b*(-C6 + 2*C7*h - 3*C8*h^2)
                         C11*b^2/2 + 2*C2 + C7*b
                                           12⋅C₄
                               6⋅C₃ + 3⋅C₈⋅b + q

In [74]:
s1 = σx(x => -b / 2)
eqs = vcat(eqs, coeff_equations(s1))

10-element Vector{Sym{PyCall.PyObject}}:
                          8⋅C₁₁ + 24⋅C₁₄ + 24⋅C₄
 C14*b^3 + b*(-2*C10*h + 2*C11*h^2 + 2*C9 - h*q)
                         b^3*(2*C12 - 2*C13*h)/4
        -C13*b^3/4 + b*(-C6 + 2*C7*h - 3*C8*h^2)
                         C11*b^2/2 + 2*C2 + C7*b
                                           12⋅C₄
                               6⋅C₃ + 3⋅C₈⋅b + q
                         C11*b^2/2 + 2*C2 - C7*b
                                           12⋅C₄
                               6⋅C₃ - 3⋅C₈⋅b + q

In [75]:
s1 = τxy(x => b / 2)
eqs = vcat(eqs, coeff_equations(s1))

13-element Vector{Sym{PyCall.PyObject}}:
                          8⋅C₁₁ + 24⋅C₁₄ + 24⋅C₄
 C14*b^3 + b*(-2*C10*h + 2*C11*h^2 + 2*C9 - h*q)
                         b^3*(2*C12 - 2*C13*h)/4
        -C13*b^3/4 + b*(-C6 + 2*C7*h - 3*C8*h^2)
                         C11*b^2/2 + 2*C2 + C7*b
                                           12⋅C₄
                               6⋅C₃ + 3⋅C₈⋅b + q
                         C11*b^2/2 + 2*C2 - C7*b
                                           12⋅C₄
                               6⋅C₃ - 3⋅C₈⋅b + q
                       -C10*b - 3*C13*b^2/4 - C6
                                           -3⋅C₈
                                 -2⋅C₁₁⋅b - 2⋅C₇

In [76]:
s1 = τxy(x => -b / 2)
eqs = vcat(eqs, coeff_equations(s1))

16-element Vector{Sym{PyCall.PyObject}}:
                          8⋅C₁₁ + 24⋅C₁₄ + 24⋅C₄
 C14*b^3 + b*(-2*C10*h + 2*C11*h^2 + 2*C9 - h*q)
                         b^3*(2*C12 - 2*C13*h)/4
        -C13*b^3/4 + b*(-C6 + 2*C7*h - 3*C8*h^2)
                         C11*b^2/2 + 2*C2 + C7*b
                                           12⋅C₄
                               6⋅C₃ + 3⋅C₈⋅b + q
                         C11*b^2/2 + 2*C2 - C7*b
                                           12⋅C₄
                               6⋅C₃ - 3⋅C₈⋅b + q
                       -C10*b - 3*C13*b^2/4 - C6
                                           -3⋅C₈
                                 -2⋅C₁₁⋅b - 2⋅C₇
                        C10*b - 3*C13*b^2/4 - C6
                                           -3⋅C₈
                                  2⋅C₁₁⋅b - 2⋅C₇

In [77]:
sol = solve(eqs, coeffs)

Dict{Sym{PyCall.PyObject}, Sym{PyCall.PyObject}} with 12 entries:
  C11 => 0
  C3  => -q/6
  C6  => 0
  C2  => 0
  C10 => 0
  C13 => 0
  C8  => 0
  C9  => h*q/2
  C4  => 0
  C12 => 0
  C7  => 0
  C14 => 0

In [78]:
ϕ_sol = subs(ϕ, sol)

                        2      3
                   h⋅q⋅x    q⋅y 
C₀ + C₁⋅y + C₅⋅x + ────── - ────
                     2       6  

In [79]:
σx = sx(ϕ_sol) + V
factor(σx)

0

In [80]:
σy = sy(ϕ_sol) + V

h⋅q + q⋅y

In [81]:
τxy = sxy(ϕ_sol)
simplify(τxy)

0